# Level 0 - The body as a tank of water
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YOUR-USERNAME/kidney-sim/blob/main/notebooks/00_body_tank.ipynb)

**Goal:** understand *why* we need kidneys with the simplest possible model.

**You will learn:** mass balance, clearance, steady state, Euler's method.

In [ ]:
# --- Setup: run this cell first --------------------------------------------
import os, sys, subprocess
REPO_URL = "https://github.com/YOUR-USERNAME/kidney-sim.git"   # change if you forked the project
if "google.colab" in sys.modules and not os.path.exists("kidneysim"):
    if not os.path.exists("kidney-sim"):
        subprocess.run(["git", "clone", "-q", REPO_URL], check=True)
    os.chdir("kidney-sim")
for path in [".", ".."]:
    if os.path.exists(os.path.join(path, "kidneysim")):
        sys.path.insert(0, os.path.abspath(path))
        break

import numpy as np
import matplotlib.pyplot as plt
import kidneysim as ks
from kidneysim import viz2d, viz3d
from kidneysim.ui import sliders
print("kidneysim", ks.__version__, "| interactive 3D (plotly):", viz3d.HAS_PLOTLY)

## 1. The idea

The liver produces **urea** all the time (it is how we get rid of the nitrogen from proteins).
Urea dissolves in all the water of the body, about **42 L**. The kidneys remove it.

Think of a bathtub: the tap is the liver, the drain is the kidney.

## 2. The governing equation (mass balance)

$$ V\,\frac{dC}{dt} = G - K\,C $$

| symbol | meaning | normal value |
|---|---|---|
| $C$ | plasma urea concentration | 5 mmol/L |
| $V$ | body water | 42 L |
| $G$ | urea made by the liver | 0.28 mmol/min |
| $K$ | **clearance**: litres of plasma completely cleaned per minute | 0.056 L/min |

When nothing changes any more ($dC/dt = 0$) we reach the **steady state**:
$$ C_{ss} = \frac{G}{K} $$

## 3. Solving it with Euler's method

A computer cannot do calculus directly, so we take small time steps $\Delta t$:

$$ C(t+\Delta t) = C(t) + \Delta t\,\frac{G - K\,C(t)}{V} $$

In [ ]:
# Euler's method written out by hand
V = 42.0        # L
G = 0.28        # mmol/min
K = 0.056       # L/min  (healthy kidneys)
dt = 10.0       # min
C = 5.0         # mmol/L, starting value

times, values = [0.0], [C]
for step in range(int(14 * 1440 / dt)):      # 14 days
    C = C + dt * (G - K * C) / V
    times.append((step + 1) * dt / 1440)     # in days
    values.append(C)

print(f"after 14 days: C = {C:.2f} mmol/L,  steady state G/K = {G/K:.2f} mmol/L")

## 4. What happens if the kidneys fail?

The package function `ks.body.simulate_waste` does exactly the loop above.
Let's compare healthy kidneys, half of the kidney (e.g. after donating one kidney) and kidney failure.

In [ ]:
runs = []
for f in [1.0, 0.5, 0.15]:
    t, C = ks.body.simulate_waste(days=14, kidney_function=f)
    runs.append((f"{int(f*100)} % kidney function", t, C))
viz2d.plot_tank(runs);

### Try it yourself
Move the slider. Creatinine (from muscles) is the substance doctors use to estimate kidney function.

In [ ]:
def experiment(kidney_function=1.0, solute=["urea", "creatinine"]):
    t, C = ks.body.simulate_waste(days=21, kidney_function=kidney_function, solute=solute)
    viz2d.plot_tank([(f"{kidney_function:.0%} function", t, C)], solute=solute)
    plt.show()

sliders(experiment, kidney_function=(0.05, 1.0, 0.05, 1.0), solute=["urea", "creatinine"])

## 5. Questions
1. If kidney function halves, what happens to the steady-state urea? Can you explain it using $C_{ss}=G/K$?
2. Why does it take *days* to reach the new value? (Hint: the time constant is $\tau = V/K$.)
3. A high-protein diet increases $G$. What happens to urea?

## What is missing?
We just *chose* $K$. But **how** does the kidney clean the blood? Next level: the filter.